# 00 — Understand Water Bodies

Predict how reflectance becomes a water mask, then observe each stage using the real commands.

Run this lab in the **dev-platform** workspace on Minikube or Kubernetes, using the `eoepca/pde-code-server:amd64` image. Select its preinstalled **Bash** kernel and use **Shift+Enter**, reading each explanation before executing its cell. No terminal or kernel installation is needed. Stop at an unexpected error and fix it before continuing. The course environment is provisioned by workspace startup.

The kernel keeps environment variables and the working directory between cells. The first cell finds the course checkout and activates its locked environment. Command groups run in a subshell with `set -euo pipefail` so a failing command stops that cell without killing the kernel. Scratch results stay in `.notebook-runs/00-waterbodies/`; rerunning a cell replaces only its lab outputs.

[Lesson](../modules/00-waterbodies/lesson.md) · [Exercise](../modules/00-waterbodies/exercise.md)


## Prepare the notebook workspace

Run this cell after every kernel restart. It clears inherited `CWLTOOL_OPTIONS` so a Coder workspace configured for Podman can still run the explicitly local exercises. It does not change your cluster or publish anything.


In [ ]:
lab_start_dir="$PWD"
while [[ ! -f reference/waterbodies.cwl || ! -f pyproject.toml ]]; do
    if [[ "$PWD" == / ]]; then
        cd "$lab_start_dir"
        break
    fi
    cd ..
done
export WORKSPACE="$PWD"
if [[ -x "$WORKSPACE/.venv/bin/python" && -f "$WORKSPACE/reference/waterbodies.cwl" ]]; then
    source "$WORKSPACE/.venv/bin/activate"
    export PYTHONPATH="$WORKSPACE${PYTHONPATH:+:$PYTHONPATH}"
    export CWLTOOL_OPTIONS=
    export MPLBACKEND=Agg
    export LAB_DIR="$WORKSPACE/.notebook-runs/00-waterbodies"
    mkdir -p "$LAB_DIR"
    cd "$LAB_DIR"
    python --version
    waterbodies --help
else
    printf 'Open this notebook inside the course checkout. Ask the instructor to check workspace environment provisioning.\n' >&2
    false
fi


## Predict the spectral signal

The synthetic scene has two equal halves: green/NIR reflectance of 100/300 on the left and 300/100 on the right. Predict `(green - nir) / (green + nir)` for each half. Which sign should represent water?


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
from scripts.fixtures import create_fixture
create_fixture(Path('source'))
print('Created an 8 x 8 STAC scene with green and NIR assets under source/')
PYTHON
)


## Execute the scientific chain

The AOI `1,1,7,7` cuts the 8×8 scene to a 6×6 grid. Two crop commands select bands through the STAC EO extension; normalized difference consumes them in green/NIR order. Otsu separates the two populations. STAC packages the result with spatial and temporal metadata.


In [ ]:
(
set -euo pipefail
waterbodies crop --input-item source --aoi "1,1,7,7" --epsg EPSG:4326 --band green
waterbodies crop --input-item source --aoi "1,1,7,7" --epsg EPSG:4326 --band nir
waterbodies norm-diff --rasters crop_green.tif --rasters crop_nir.tif
waterbodies otsu --raster norm_diff.tif
waterbodies stac --input-item source --water-body otsu.tif
)


## View the bands, index and mask in the notebook

The Python heredoc inspects the rasters without changing kernels. Bash's `display` function embeds the saved figure in the cell output. Predict the boundary before running.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
import matplotlib.pyplot as plt
import numpy as np
import rasterio
figure, axes = plt.subplots(1, 4, figsize=(14, 3))
for axis, filename in zip(axes, ['crop_green.tif', 'crop_nir.tif', 'norm_diff.tif', 'otsu.tif'], strict=True):
    with rasterio.open(filename) as raster:
        pixels = raster.read(1)
        print(filename, raster.shape, raster.dtypes[0], np.unique(pixels))
        axis.imshow(pixels)
        axis.set_title(filename)
figure.tight_layout()
figure.savefig('chain.png')
with rasterio.open('otsu.tif') as raster:
    assert np.count_nonzero(raster.read(1)) == 18
PYTHON
display "" < chain.png
)


## Validate the product

Read the packaged Item through PySTAC, resolve its mask asset and validate the declared extensions. A JSON file alone does not establish a valid STAC product.


In [ ]:
(
set -euo pipefail
python - <<'PYTHON'
from pathlib import Path
from waterbodies.common import read_item
from waterbodies.validation import validate_item
item = read_item(Path('catalog'))
validate_item(item)
print(item.id, item.datetime, item.bbox)
print('Mask:', item.assets['data'].get_absolute_href())
print('STAC validation passed')
PYTHON
)


## Experiment

Change both crop AOIs to `1,1,4,7`, predict the effect and rerun the chain and plot. Then restore `1,1,7,7`. The existing [Python exploration notebook](../modules/00-waterbodies/waterbodies.ipynb) offers additional graphical experiments.


## Record your findings

**Expected outcome:** The default crops are 6×6, NDWI is −0.5/+0.5, the mask has 18 water pixels and STAC validation passes.

Add your observations to a Markdown cell: what you predicted, what the command produced, and which contract declaration explains it. Restart the kernel and run all cells with the default settings to check reproducibility.

Compare [the solution](../modules/00-waterbodies/solution.md) after the activity. Return to [the lab index](README.md).
